# Notebook 5: Image Captioning (Generación de Descripciones)

**Autores:** Javier Arroyo | Julia Cano | Paula Durá  
**Asignatura:** Procesamiento de Imágenes  

---

## Objetivo

Generar automáticamente **descripciones textuales en lenguaje natural** para las imágenes del dataset. A diferencia de la clasificación (una etiqueta) o la detección (objetos + localización), aquí el modelo debe producir una **frase completa** que capture el contenido semántico de la imagen.

### Modelos implementados

| # | Modelo | Enfoque | Arquitectura |
|---|--------|---------|-------------|
| 1 | BLIP (Salesforce) | Preentrenado | ViT encoder + Transformer decoder |
| 2 | ViT-GPT2 | Preentrenado | Vision Transformer + GPT-2 autoregresivo |
| 3 | CNN+LSTM (from scratch) | Entrenado desde cero | ResNet18 features + LSTM decoder |

### Evaluación
- **BLEU Score** (Bilingual Evaluation Understudy): similitud n-gram con captions de referencia
- **Análisis cualitativo**: coherencia, detalle y vocabulario de los captions generados
- **Análisis de vocabulario por categoría**: palabras más frecuentes por clase

> **Nota:** Se usa el dataset aumentado del Notebook 01. Los captions de BLIP sirven como pseudo-referencia para evaluar los otros modelos.

---
## 5.1 Configuración

Importamos `transformers` (Hugging Face) para los modelos preentrenados y TensorFlow/Keras para el modelo from scratch.

In [ ]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter
import random
import warnings
warnings.filterwarnings("ignore")

from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torchvision import transforms, models

from transformers import (
    BlipProcessor, BlipForConditionalGeneration,
    VisionEncoderDecoderModel, ViTImageProcessor, AutoTokenizer
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

In [ ]:
# Cargar dataset
DATA_DIR = Path("dataset_augmented")
if not DATA_DIR.exists():
    DATA_DIR = Path("dataset")

classes = sorted([p.name for p in DATA_DIR.iterdir() if p.is_dir()])
print("Clases:", classes)

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
rows = []
for cls in classes:
    for p in (DATA_DIR / cls).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMG_EXTS:
            rows.append({"path": str(p), "class": cls})

df = pd.DataFrame(rows)
print(f"Total imágenes: {len(df)}")

# Seleccionar subconjunto para captioning (10 por clase para rapidez)
sample_df = df.groupby("class").apply(lambda x: x.sample(min(10, len(x)), random_state=SEED)).reset_index(drop=True)
print(f"Muestra para captioning: {len(sample_df)}")

---
## 5.2 Modelo Preentrenado 1: BLIP (Salesforce)

### Arquitectura

**BLIP** (*Bootstrapping Language-Image Pre-training*) es un modelo de visión-lenguaje diseñado para múltiples tareas (captioning, VQA, retrieval). Su arquitectura incluye:

- **Vision Transformer (ViT):** Encoder visual que divide la imagen en *patches* y los procesa con self-attention
- **Transformer Decoder:** Genera texto autoregressivamente, condicionado en las features visuales
- **Preentrenamiento:** Millones de pares imagen-texto + técnica de *bootstrapping* para filtrar datos ruidosos

El modelo `blip-image-captioning-base` tiene ~247M de parámetros y ha sido entrenado en COCO Captions, Visual Genome, y datos web filtrados.

In [ ]:
# Cargar BLIP para captioning
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
blip_model.eval()

print("BLIP model loaded successfully")
print(f"Parámetros: {sum(p.numel() for p in blip_model.parameters()) / 1e6:.1f}M")

In [ ]:
def generate_caption_blip(img_path, max_length=50):
    """Genera un caption usando BLIP."""
    img = Image.open(img_path).convert("RGB")
    inputs = blip_processor(images=img, return_tensors="pt").to(device)
    
    with torch.no_grad():
        output = blip_model.generate(**inputs, max_length=max_length, num_beams=5)
    
    caption = blip_processor.decode(output[0], skip_special_tokens=True)
    return caption

# Test con una imagen
test_path = sample_df["path"].iloc[0]
caption = generate_caption_blip(test_path)
print(f"Imagen: {Path(test_path).name}")
print(f"Caption: {caption}")

### 5.2.1 Generación masiva de captions con BLIP

Generamos captions para todas las imágenes de la muestra usando *beam search* (5 beams) para maximizar la calidad de las descripciones.

In [ ]:
# Generar captions para todas las imágenes de la muestra
blip_captions = []

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="BLIP captions"):
    try:
        caption = generate_caption_blip(row["path"])
        blip_captions.append({
            "path": row["path"],
            "class": row["class"],
            "caption_blip": caption
        })
    except Exception as e:
        blip_captions.append({
            "path": row["path"],
            "class": row["class"],
            "caption_blip": f"[Error: {str(e)[:50]}]"
        })

blip_df = pd.DataFrame(blip_captions)
print(f"Captions generados: {len(blip_df)}")

In [ ]:
# Visualizar captions BLIP por categoría (2 imágenes por clase)
fig, axes = plt.subplots(len(classes), 2, figsize=(14, 5 * len(classes)))

for i, cls in enumerate(classes):
    cls_data = blip_df[blip_df["class"] == cls].head(2)
    for j, (_, row) in enumerate(cls_data.iterrows()):
        img = Image.open(row["path"]).convert("RGB")
        axes[i, j].imshow(img)
        axes[i, j].set_title(f"[{cls}]\n{row['caption_blip']}", fontsize=9, wrap=True)
        axes[i, j].axis("off")

plt.suptitle("BLIP: Captions generados por categoría", fontsize=14)
plt.tight_layout()
plt.show()

---
## 5.3 Modelo Preentrenado 2: ViT-GPT2

### Arquitectura

**ViT-GPT2** combina dos modelos especializados mediante un *encoder-decoder framework*:

- **ViT (Vision Transformer):** Extrae representaciones visuales ricas procesando la imagen como secuencia de *patches* 16×16
- **GPT-2 (Generative Pre-trained Transformer 2):** Decoder autoregresivo que genera texto token a token, condicionado en la representación visual

Este enfoque modular permite aprovechar el preentrenamiento independiente de cada componente.

In [ ]:
# Cargar ViT-GPT2
vit_gpt2_model = VisionEncoderDecoderModel.from_pretrained("nlpconnect/vit-gpt2-image-captioning").to(device)
vit_gpt2_processor = ViTImageProcessor.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
vit_gpt2_tokenizer = AutoTokenizer.from_pretrained("nlpconnect/vit-gpt2-image-captioning")
vit_gpt2_model.eval()

print("ViT-GPT2 model loaded successfully")
print(f"Parámetros: {sum(p.numel() for p in vit_gpt2_model.parameters()) / 1e6:.1f}M")

In [ ]:
def generate_caption_vitgpt2(img_path, max_length=50):
    """Genera un caption usando ViT-GPT2."""
    img = Image.open(img_path).convert("RGB")
    pixel_values = vit_gpt2_processor(images=img, return_tensors="pt").pixel_values.to(device)
    
    with torch.no_grad():
        output = vit_gpt2_model.generate(pixel_values, max_length=max_length, num_beams=5)
    
    caption = vit_gpt2_tokenizer.decode(output[0], skip_special_tokens=True)
    return caption

# Test
caption_vg = generate_caption_vitgpt2(test_path)
print(f"ViT-GPT2 caption: {caption_vg}")

In [ ]:
# Generar captions ViT-GPT2 para toda la muestra
vitgpt2_captions = []

for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="ViT-GPT2 captions"):
    try:
        caption = generate_caption_vitgpt2(row["path"])
        vitgpt2_captions.append({
            "path": row["path"],
            "class": row["class"],
            "caption_vitgpt2": caption
        })
    except Exception as e:
        vitgpt2_captions.append({
            "path": row["path"],
            "class": row["class"],
            "caption_vitgpt2": f"[Error: {str(e)[:50]}]"
        })

vitgpt2_df = pd.DataFrame(vitgpt2_captions)

### 5.3.1 Comparación visual: BLIP vs ViT-GPT2

Mostramos los captions de ambos modelos lado a lado para observar diferencias en estilo, detalle y terminología utilizada.

In [ ]:
# Combinar captions de ambos modelos
combined_df = blip_df.merge(vitgpt2_df[["path", "caption_vitgpt2"]], on="path", how="left")

# Mostrar comparación (1 imagen por clase)
fig, axes = plt.subplots(len(classes), 1, figsize=(12, 5 * len(classes)))

for i, cls in enumerate(classes):
    row = combined_df[combined_df["class"] == cls].iloc[0]
    img = Image.open(row["path"]).convert("RGB")
    axes[i].imshow(img)
    axes[i].set_title(
        f"[{cls}]\n"
        f"BLIP:     {row['caption_blip']}\n"
        f"ViT-GPT2: {row['caption_vitgpt2']}",
        fontsize=10, loc="left"
    )
    axes[i].axis("off")

plt.suptitle("Comparación de captions: BLIP vs ViT-GPT2", fontsize=14)
plt.tight_layout()
plt.show()

---
## 5.4 Análisis semántico de captions por categoría

Analizamos las **palabras más frecuentes** en los captions generados para cada categoría. Si los modelos capturan correctamente la semántica visual, las palabras dominantes deben reflejar el contenido típico de cada clase (e.g., "dog"/"cat" para Animales, "building"/"street" para Ciudad).

In [ ]:
import re

def get_word_freq(texts, top_n=15):
    """Obtiene frecuencia de palabras (sin stopwords básicas)."""
    stopwords = {"a", "an", "the", "of", "in", "on", "at", "to", "and", "is", "with", "for", "it", "by", "as", "are", "its"}
    words = []
    for text in texts:
        tokens = re.findall(r'\b[a-zA-Z]{2,}\b', text.lower())
        words.extend([w for w in tokens if w not in stopwords])
    freq = Counter(words)
    return freq.most_common(top_n)

# Análisis por clase (BLIP)
fig, axes = plt.subplots(1, len(classes), figsize=(5 * len(classes), 5))

for i, cls in enumerate(classes):
    cls_captions = combined_df[combined_df["class"] == cls]["caption_blip"].tolist()
    freq = get_word_freq(cls_captions, top_n=10)
    
    if freq:
        words, counts = zip(*freq)
        axes[i].barh(range(len(words)), counts, color=plt.cm.Set2(i / len(classes)))
        axes[i].set_yticks(range(len(words)))
        axes[i].set_yticklabels(words)
        axes[i].invert_yaxis()
        axes[i].set_title(f"{cls}", fontsize=12)
        axes[i].set_xlabel("Frecuencia")

plt.suptitle("Palabras más frecuentes en captions BLIP por categoría", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Longitud media de captions por modelo y categoría
combined_df["len_blip"] = combined_df["caption_blip"].str.split().str.len()
combined_df["len_vitgpt2"] = combined_df["caption_vitgpt2"].str.split().str.len()

len_stats = combined_df.groupby("class")[["len_blip", "len_vitgpt2"]].mean()

fig, ax = plt.subplots(figsize=(10, 5))
len_stats.plot(kind="bar", ax=ax, color=["#3498db", "#e74c3c"], alpha=0.8)
ax.set_title("Longitud media de captions por modelo y categoría")
ax.set_ylabel("Número medio de palabras")
ax.set_xlabel("Categoría")
ax.tick_params(axis='x', rotation=45)
ax.legend(["BLIP", "ViT-GPT2"])
plt.tight_layout()
plt.show()

---
## 5.5 Modelo From Scratch: CNN Encoder + LSTM Decoder

### Arquitectura

Implementamos un **captioner clásico** basado en la arquitectura *Show and Tell* (Vinyals et al., 2015):

1. **Encoder (CNN):** ResNet18 preentrenada (congelada) extrae un vector de features $\mathbf{v} \in \mathbb{R}^{512}$ por imagen
2. **Decoder (LSTM):** Genera la descripción palabra a palabra usando *teacher forcing* durante el entrenamiento

### Limitaciones esperadas
- **Dataset minúsculo:** ~150 pares imagen-caption (frente a los 330K+ de COCO usados por los preentrenados)
- **Vocabulario muy reducido**: pocas palabras únicas en el corpus de entrenamiento
- **Sin atención:** No implementa mecanismo de atención sobre regiones de la imagen

Este ejercicio sirve para entender la mecánica de la generación de texto condicionada en imágenes y apreciar el valor del preentrenamiento masivo.

In [ ]:
# Paso 1: Generar captions de referencia con BLIP para TODO el dataset (subconjunto mayor)
# Usamos 30 imágenes por clase para tener más datos de entrenamiento
train_sample = df.groupby("class").apply(lambda x: x.sample(min(30, len(x)), random_state=SEED)).reset_index(drop=True)

print(f"Generando captions de referencia para {len(train_sample)} imágenes...")

ref_captions = []
for idx, row in tqdm(train_sample.iterrows(), total=len(train_sample), desc="Generating reference captions"):
    try:
        caption = generate_caption_blip(row["path"])
        ref_captions.append({"path": row["path"], "class": row["class"], "caption": caption})
    except:
        ref_captions.append({"path": row["path"], "class": row["class"], "caption": f"a photo of {row['class']}"})

ref_df = pd.DataFrame(ref_captions)
print(f"Captions de referencia: {len(ref_df)}")
print("Ejemplo:", ref_df["caption"].iloc[0])

In [ ]:
# Paso 2: Construir vocabulario y tokenizar
class Vocabulary:
    def __init__(self, freq_threshold=1):
        self.word2idx = {"<PAD>": 0, "<SOS>": 1, "<EOS>": 2, "<UNK>": 3}
        self.idx2word = {0: "<PAD>", 1: "<SOS>", 2: "<EOS>", 3: "<UNK>"}
        self.freq_threshold = freq_threshold
    
    def build(self, captions):
        word_freq = Counter()
        for cap in captions:
            tokens = cap.lower().split()
            word_freq.update(tokens)
        
        idx = len(self.word2idx)
        for word, freq in word_freq.items():
            if freq >= self.freq_threshold:
                self.word2idx[word] = idx
                self.idx2word[idx] = word
                idx += 1
        print(f"Vocabulario: {len(self.word2idx)} palabras")
    
    def encode(self, caption, max_len=20):
        tokens = [self.word2idx.get(w, self.word2idx["<UNK>"]) for w in caption.lower().split()]
        tokens = [self.word2idx["<SOS>"]] + tokens[:max_len-2] + [self.word2idx["<EOS>"]]
        # Pad
        tokens += [self.word2idx["<PAD>"]] * (max_len - len(tokens))
        return tokens[:max_len]
    
    def decode(self, indices):
        words = []
        for idx in indices:
            if idx == self.word2idx["<EOS>"]:
                break
            if idx not in (self.word2idx["<PAD>"], self.word2idx["<SOS>"]):
                words.append(self.idx2word.get(idx, "<UNK>"))
        return " ".join(words)

vocab = Vocabulary(freq_threshold=1)
vocab.build(ref_df["caption"].tolist())

MAX_LEN = 20

In [ ]:
# Paso 3: Extraer features con ResNet18 (congelada)
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet = nn.Sequential(*list(resnet.children())[:-1])  # Quitar la última capa FC
resnet.eval()
resnet.to(device)

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def extract_features(img_path):
    img = Image.open(img_path).convert("RGB")
    img_tensor = preprocess(img).unsqueeze(0).to(device)
    with torch.no_grad():
        features = resnet(img_tensor).squeeze()  # [512]
    return features.cpu().numpy()

# Extraer features para todo el conjunto
print("Extrayendo features con ResNet18...")
all_features = []
all_captions_encoded = []

for idx, row in tqdm(ref_df.iterrows(), total=len(ref_df), desc="Feature extraction"):
    try:
        feat = extract_features(row["path"])
        cap_encoded = vocab.encode(row["caption"], MAX_LEN)
        all_features.append(feat)
        all_captions_encoded.append(cap_encoded)
    except:
        pass

X_features = np.array(all_features)  # [N, 512]
Y_captions = np.array(all_captions_encoded)  # [N, MAX_LEN]

print(f"Features shape: {X_features.shape}")
print(f"Captions shape: {Y_captions.shape}")

In [ ]:
# Paso 4: Modelo from scratch con Keras (Image Features -> Caption)
EMBED_DIM = 64
HIDDEN_DIM = 128
VOCAB_SIZE = len(vocab.word2idx)

# Encoder: proyecta features de la imagen a un espacio compartido
feature_input = keras.Input(shape=(512,), name="image_features")
feat_proj = layers.Dense(HIDDEN_DIM, activation="relu")(feature_input)

# Decoder input: secuencia de tokens (teacher forcing)
caption_input = keras.Input(shape=(MAX_LEN,), dtype="int32", name="caption_input")
embed = layers.Embedding(VOCAB_SIZE, EMBED_DIM, mask_zero=True)(caption_input)

# Combinar: repetir features + concatenar con embeddings
feat_repeated = layers.RepeatVector(MAX_LEN)(feat_proj)  # [batch, MAX_LEN, HIDDEN_DIM]
combined = layers.Concatenate()([feat_repeated, embed])   # [batch, MAX_LEN, HIDDEN_DIM + EMBED_DIM]

# LSTM decoder
x = layers.LSTM(HIDDEN_DIM, return_sequences=True)(combined)
x = layers.Dropout(0.3)(x)
x = layers.LSTM(HIDDEN_DIM, return_sequences=True)(x)
x = layers.Dropout(0.3)(x)

# Output: probabilidad sobre vocabulario para cada posición
output = layers.TimeDistributed(layers.Dense(VOCAB_SIZE, activation="softmax"))(x)

caption_model = keras.Model([feature_input, caption_input], output, name="cnn_lstm_captioner")
caption_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

caption_model.summary()

In [ ]:
# Preparar datos para entrenamiento
# Input: features + caption[:-1] (teacher forcing)
# Target: caption[1:] (siguiente palabra)

X_feat_train = X_features
X_cap_train = Y_captions[:, :-1]   # todo menos el último token
Y_target = Y_captions[:, 1:]       # todo menos el primer token (shifted)

# Expandir target a [batch, seq_len, 1] para sparse_categorical_crossentropy
Y_target = Y_target[..., np.newaxis]

# Pad caption input to be same length MAX_LEN
X_cap_padded = np.zeros((len(X_cap_train), MAX_LEN), dtype=np.int32)
X_cap_padded[:, :X_cap_train.shape[1]] = X_cap_train

Y_target_padded = np.zeros((len(Y_target), MAX_LEN, 1), dtype=np.int32)
Y_target_padded[:, :Y_target.shape[1], :] = Y_target

print(f"X_feat: {X_feat_train.shape}, X_cap: {X_cap_padded.shape}, Y_target: {Y_target_padded.shape}")

In [ ]:
# Entrenar
from sklearn.model_selection import train_test_split

indices = np.arange(len(X_feat_train))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=SEED)

history_cap = caption_model.fit(
    [X_feat_train[train_idx], X_cap_padded[train_idx]],
    Y_target_padded[train_idx],
    validation_data=([X_feat_train[val_idx], X_cap_padded[val_idx]], Y_target_padded[val_idx]),
    epochs=50,
    batch_size=16,
    callbacks=[keras.callbacks.EarlyStopping(patience=8, restore_best_weights=True)],
    verbose=1
)

### 5.5.1 Curvas de entrenamiento

Monitorizamos loss y accuracy por token para verificar que el LSTM aprende patrones de secuencia, incluso con datos limitados.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history_cap.history["loss"], label="train_loss")
axes[0].plot(history_cap.history["val_loss"], label="val_loss")
axes[0].set_title("Caption Model - Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history_cap.history["accuracy"], label="train_acc")
axes[1].plot(history_cap.history["val_accuracy"], label="val_acc")
axes[1].set_title("Caption Model - Accuracy (per token)")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.suptitle("CNN+LSTM Captioner from scratch - Entrenamiento", fontsize=13)
plt.tight_layout()
plt.show()

### 5.5.2 Generación de captions (*greedy decoding*)

Para generar captions en inferencia, usamos *greedy decoding*: en cada paso temporal, seleccionamos la palabra con mayor probabilidad. Esto es más simple (pero menos óptimo) que *beam search*.

In [ ]:
def generate_caption_scratch(img_path, model, vocab, max_len=MAX_LEN):
    """Genera un caption con el modelo from scratch (greedy decoding)."""
    feat = extract_features(img_path)
    feat = feat[np.newaxis, :]  # [1, 512]
    
    # Empezar con <SOS>
    caption_seq = np.zeros((1, max_len), dtype=np.int32)
    caption_seq[0, 0] = vocab.word2idx["<SOS>"]
    
    for t in range(1, max_len):
        preds = model.predict([feat, caption_seq], verbose=0)  # [1, max_len, vocab_size]
        next_word_idx = np.argmax(preds[0, t-1, :])
        caption_seq[0, t] = next_word_idx
        
        if next_word_idx == vocab.word2idx["<EOS>"]:
            break
    
    return vocab.decode(caption_seq[0])

# Generar captions scratch para la muestra
scratch_captions = []
for idx, row in tqdm(sample_df.iterrows(), total=len(sample_df), desc="Scratch captions"):
    try:
        caption = generate_caption_scratch(row["path"], caption_model, vocab)
        scratch_captions.append({"path": row["path"], "class": row["class"], "caption_scratch": caption})
    except Exception as e:
        scratch_captions.append({"path": row["path"], "class": row["class"], "caption_scratch": f"[Error]"})

scratch_df = pd.DataFrame(scratch_captions)

---
## 5.6 Comparación final: tres modelos

Mostramos los captions de los tres modelos lado a lado para cada categoría, seguido de evaluación cuantitativa con BLEU.

In [ ]:
# Unir todos los captions
final_df = combined_df.merge(scratch_df[["path", "caption_scratch"]], on="path", how="left")

# Mostrar comparación lado a lado
print("=" * 100)
print("COMPARACIÓN DE CAPTIONS POR MODELO")
print("=" * 100)

for cls in classes:
    cls_data = final_df[final_df["class"] == cls].head(2)
    print(f"\n{'─'*80}")
    print(f"  Categoría: {cls.upper()}")
    print(f"{'─'*80}")
    for _, row in cls_data.iterrows():
        print(f"  BLIP:     {row['caption_blip']}")
        print(f"  ViT-GPT2: {row['caption_vitgpt2']}")
        print(f"  Scratch:  {row.get('caption_scratch', 'N/A')}")
        print()

In [ ]:
# Visualización comparativa (1 imagen por clase, 3 modelos)
fig, axes = plt.subplots(len(classes), 1, figsize=(14, 5 * len(classes)))

for i, cls in enumerate(classes):
    row = final_df[final_df["class"] == cls].iloc[0]
    img = Image.open(row["path"]).convert("RGB")
    axes[i].imshow(img)
    
    caption_text = (
        f"[{cls}]\n"
        f"  BLIP:      {row['caption_blip']}\n"
        f"  ViT-GPT2:  {row['caption_vitgpt2']}\n"
        f"  Scratch:   {row.get('caption_scratch', 'N/A')}"
    )
    axes[i].set_title(caption_text, fontsize=10, loc="left", family="monospace")
    axes[i].axis("off")

plt.suptitle("Comparación de Image Captioning: BLIP vs ViT-GPT2 vs From Scratch", fontsize=14)
plt.tight_layout()
plt.show()

### 5.6.1 Evaluación cuantitativa: BLEU Score

**BLEU** mide la superposición de n-gramas entre el texto generado y una referencia. Usamos BLEU-2 (bigramas) como compromiso entre precisión y cobertura.

$$\text{BLEU} = BP \cdot \exp\left(\sum_{n=1}^{N} \frac{1}{N} \log p_n\right)$$

donde $p_n$ es la precisión de n-gramas y $BP$ es una penalización por brevedad.

> **Nota:** Usamos captions de BLIP como pseudo-referencia (al no disponer de anotaciones humanas).

In [ ]:
from collections import Counter

def compute_bleu_ngram(reference, candidate, n=4):
    """Calcula BLEU-n simplificado entre dos textos."""
    ref_tokens = reference.lower().split()
    cand_tokens = candidate.lower().split()
    
    if len(cand_tokens) == 0:
        return 0.0
    
    # Brevity penalty
    bp = min(1.0, np.exp(1 - len(ref_tokens) / max(len(cand_tokens), 1)))
    
    precisions = []
    for i in range(1, n + 1):
        ref_ngrams = Counter([tuple(ref_tokens[j:j+i]) for j in range(len(ref_tokens) - i + 1)])
        cand_ngrams = Counter([tuple(cand_tokens[j:j+i]) for j in range(len(cand_tokens) - i + 1)])
        
        clipped = sum(min(cand_ngrams[ng], ref_ngrams[ng]) for ng in cand_ngrams)
        total = sum(cand_ngrams.values())
        
        if total == 0:
            precisions.append(0)
        else:
            precisions.append(clipped / total)
    
    # Geometric mean de las precisions
    if any(p == 0 for p in precisions):
        return 0.0
    
    log_avg = sum(np.log(p) for p in precisions) / n
    return bp * np.exp(log_avg)

# Calcular BLEU scores
bleu_vitgpt2 = []
bleu_scratch = []

for _, row in final_df.iterrows():
    ref = row["caption_blip"]
    
    # ViT-GPT2 vs BLIP
    cand_vg = row.get("caption_vitgpt2", "")
    if cand_vg and not cand_vg.startswith("[Error"):
        bleu_vitgpt2.append(compute_bleu_ngram(ref, cand_vg, n=2))  # BLEU-2
    
    # Scratch vs BLIP
    cand_sc = row.get("caption_scratch", "")
    if cand_sc and not cand_sc.startswith("[Error"):
        bleu_scratch.append(compute_bleu_ngram(ref, cand_sc, n=2))

print("BLEU-2 Scores (vs BLIP como referencia):")
print(f"  ViT-GPT2:      {np.mean(bleu_vitgpt2):.4f} ± {np.std(bleu_vitgpt2):.4f}")
print(f"  From scratch:  {np.mean(bleu_scratch):.4f} ± {np.std(bleu_scratch):.4f}")

In [ ]:
# Gráfico de BLEU scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# BLEU por modelo
models_bleu = ["ViT-GPT2", "From scratch"]
means = [np.mean(bleu_vitgpt2), np.mean(bleu_scratch)]
stds = [np.std(bleu_vitgpt2), np.std(bleu_scratch)]

axes[0].bar(models_bleu, means, yerr=stds, color=["#3498db", "#e74c3c"], alpha=0.8, capsize=5)
axes[0].set_title("BLEU-2 Score por modelo (vs BLIP como ref.)")
axes[0].set_ylabel("BLEU-2")
axes[0].set_ylim(0, max(means) * 1.5 if max(means) > 0 else 1)

# Distribución de BLEU
if bleu_vitgpt2:
    axes[1].hist(bleu_vitgpt2, bins=15, alpha=0.5, color="#3498db", label="ViT-GPT2")
if bleu_scratch:
    axes[1].hist(bleu_scratch, bins=15, alpha=0.5, color="#e74c3c", label="From scratch")
axes[1].set_title("Distribución de BLEU-2 Scores")
axes[1].set_xlabel("BLEU-2")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 5.7 Conclusiones

### Resumen de rendimiento

| Modelo | Tipo | Calidad de captions | BLEU-2 | Requisitos |
|--------|------|--------------------|---------:|-----------|
| **BLIP** | Preentrenado | Excelente — detallados y coherentes | (referencia) | Solo inferencia |
| **ViT-GPT2** | Preentrenado | Muy buena — concisos y precisos | Medio-alto | Solo inferencia |
| **CNN+LSTM** | From scratch | Baja — repetitivos y genéricos | Bajo | Entrenamiento (50 epochs) |

### Hallazgos clave

1. **Los modelos preentrenados son incomparablemente superiores** en Image Captioning, una tarea que requiere comprensión visual profunda Y generación de lenguaje natural
2. **BLIP produce captions más ricos y detallados**, mientras que ViT-GPT2 tiende a descripciones más cortas y directas
3. **El modelo from scratch demuestra que la tarea es extremadamente difícil** sin preentrenamiento masivo — con ~150 ejemplos, solo aprende patrones superficiales
4. **El análisis de vocabulario confirma** que los modelos capturan correctamente la semántica de cada categoría (vocabulario específico por clase)

### Aplicaciones prácticas
- Accesibilidad (descripción para personas con discapacidad visual)
- Indexación y búsqueda de imágenes por texto
- Generación automática de metadatos para catálogos fotográficos

In [ ]:
# Guardar captions para uso en el siguiente notebook
final_df.to_csv("generated_captions.csv", index=False)
print("Captions guardados en generated_captions.csv")